# 🎓 SmartBAC - Fine-tune DeepSeek-R1-Distill-Qwen-14B cu LoRA

**Ce face acest notebook:**
- Încarcă exerciții de BAC (matematică) cu reasoning steps
- Fine-tune DeepSeek-R1-Distill-Qwen-14B cu LoRA (4-bit quantizat)
- Format `<think>...</think>` pentru chain-of-thought reasoning
- Antrenare pe GPU T4 x2
- Salvează adapterii LoRA pentru download

**Setup Kaggle:**
1. Upload `exercises_merged.json` ca Dataset pe Kaggle
2. Selectează GPU T4 x2 ca Accelerator
3. Rulează toate celulele

## 1. Instalare dependențe

In [ ]:
!pip install -q transformers>=4.40.0 peft>=0.10.0 trl>=0.8.0 bitsandbytes>=0.43.0 accelerate>=0.30.0 datasets scipy

## 2. Imports și config

In [ ]:
import json
import torch
import os
from pathlib import Path
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

In [ ]:
# ═══════════════════════════════════════════
# CONFIGURARE
# ═══════════════════════════════════════════

MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-14B"  # Model de baza
OUTPUT_DIR = "/kaggle/working/smartbac-lora"

# Calea catre dataset
DATA_PATH = "/kaggle/input/smartbac-exercises/exercises_merged.json"
if not os.path.exists(DATA_PATH):
    for p in [
        "/kaggle/input/exercises_merged.json",
        "/kaggle/input/bac-prep-data/exercises_merged.json",
        "/kaggle/input/smartbac-data/exercises_merged.json",
        "exercises_merged.json",
    ]:
        if os.path.exists(p):
            DATA_PATH = p
            break

print(f"Dataset path: {DATA_PATH}")
print(f"Exists: {os.path.exists(DATA_PATH)}")

# LoRA config
LORA_RANK = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

# Training config — ajustat pentru 14B pe T4 x2
NUM_EPOCHS = 3
BATCH_SIZE = 1           # Mic pt 14B
GRAD_ACCUM = 16          # Effective batch = 1 * 16 = 16
LEARNING_RATE = 1e-4     # Mai mic pt model mare
MAX_SEQ_LENGTH = 1024    # Mai mare pt <think> reasoning
WARMUP_RATIO = 0.1

## 3. Încărcare și pregătire dataset

In [ ]:
# Incarca exercitiile
with open(DATA_PATH, "r", encoding="utf-8") as f:
    exercises = json.load(f)

print(f"Total exercitii incarcate: {len(exercises)}")

# System prompt
SYSTEM_PROMPT = (
    "Ești SmartBAC, un asistent de matematică specializat pe exerciții de Bacalaureat. "
    "Gândește pas cu pas în blocul <think>, apoi dă răspunsul final. "
    "Răspunde în limba română."
)

def format_chatml(exercise):
    """Converteste un exercitiu in format ChatML cu <think> reasoning."""
    question = exercise.get("question", "")
    answer = exercise.get("answer", "")
    solution = exercise.get("solution", "")
    steps = exercise.get("solution_steps", [])
    
    if not question or not answer:
        return None
    
    # Construim partea de thinking (reasoning)
    thinking = ""
    if solution and len(solution.strip()) > 10:
        # Folosim solution-ul complet ca reasoning
        thinking = solution.strip()
    elif steps and len(steps) > 0:
        # Sau construim din steps
        thinking = "\n".join(s for s in steps if s and s.strip())
    else:
        # Minim: raspunsul direct
        thinking = f"Rezolvare directă: {answer}"
    
    # Format: <think>reasoning</think> + raspuns
    response = f"<think>\n{thinking}\n</think>\n\nRăspuns: {answer}"
    
    # Format ChatML
    text = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{question}<|im_end|>\n"
        f"<|im_start|>assistant\n{response}<|im_end|>"
    )
    return text

# Formateaza toate exercitiile
formatted = []
for ex in exercises:
    text = format_chatml(ex)
    if text:
        formatted.append({"text": text})

print(f"Exercitii formatate: {len(formatted)}")
print(f"\n--- Exemplu (cu <think>) ---")
print(formatted[0]["text"][:800])

In [ ]:
# Creaza dataset HuggingFace si split train/val
import random
random.seed(42)
random.shuffle(formatted)

split_idx = int(len(formatted) * 0.9)
train_data = formatted[:split_idx]
val_data = formatted[split_idx:]

train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

print(f"Train: {len(train_dataset)} samples")
print(f"Val:   {len(val_dataset)} samples")

## 4. Încărcare model (4-bit quantizat)

In [ ]:
# Quantizare 4-bit pentru a incapea pe T4 (16GB VRAM)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Incarc modelul {MODEL_NAME}...")
print("(Prima data descarca ~3GB, apoi se foloseste cache-ul)")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)

# Setam pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id

print(f"\nModel incarcat!")
print(f"Parametri totali: {model.num_parameters():,}")
print(f"Vocab size: {tokenizer.vocab_size:,}")

## 5. Configurare LoRA

In [ ]:
# Pregatim modelul pentru antrenare cu quantizare
model = prepare_model_for_kbit_training(model)

# Configurare LoRA
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    bias="none",
    task_type="CAUSAL_LM",
)

# Aplicam LoRA
model = get_peft_model(model, lora_config)

# Statistici
trainable, total = model.get_nb_trainable_parameters()
print(f"\n{'='*50}")
print(f"Parametri totali:     {total:>12,}")
print(f"Parametri antrenabili: {trainable:>12,}")
print(f"Procent antrenabil:    {100 * trainable / total:.2f}%")
print(f"{'='*50}")

## 6. Antrenare 🚀

In [ ]:
# Argumente de antrenare — optimizat pentru 14B pe T4 x2
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    fp16=True,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    max_grad_norm=0.3,
    seed=42,
    dataloader_pin_memory=False,  # Economiseste memorie
)

# SFT Trainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=training_args,
    max_seq_length=MAX_SEQ_LENGTH,
)

print(f"\n{'='*60}")
print(f"  ANTRENARE SMARTBAC (DeepSeek-R1-Distill-Qwen-14B)")
print(f"  Format: <think> chain-of-thought reasoning")
print(f"  Dataset: {len(train_dataset)} train / {len(val_dataset)} val")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Batch: {BATCH_SIZE} x {GRAD_ACCUM} = {BATCH_SIZE * GRAD_ACCUM} effective")
print(f"  LoRA rank: {LORA_RANK}, alpha: {LORA_ALPHA}")
print(f"  Max seq length: {MAX_SEQ_LENGTH}")
print(f"{'='*60}\n")

In [ ]:
# 🚀 START ANTRENARE
trainer.train()

print("\n✅ Antrenare completa!")

## 7. Salvare model

In [ ]:
# Salvam adapterii LoRA
FINAL_DIR = "/kaggle/working/smartbac-lora-final"
model.save_pretrained(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

print(f"\nAdapterii salvati in: {FINAL_DIR}")
print("\nFisiere salvate:")
for f in sorted(os.listdir(FINAL_DIR)):
    size = os.path.getsize(os.path.join(FINAL_DIR, f))
    print(f"  {f}: {size/1024:.1f} KB")

## 8. Test rapid 🧪

In [ ]:
# Test cu cateva intrebari
test_questions = [
    "Calculează derivata funcției f(x) = x³ - 3x² + 2x",
    "Rezolvă ecuația x² - 5x + 6 = 0",
    "Ce este derivata?",
    "Calculează integrala ∫(2x+1)dx",
    "Calculează limita lim(x→∞) (2x²+1)/(x²-3)",
]

print("=" * 60)
print("  TEST SMARTBAC (DeepSeek-R1 + LoRA)")
print("  Format: <think> reasoning + raspuns")
print("=" * 60)

for q in test_questions:
    prompt = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{q}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    
    print(f"\n❓ {q}")
    # Separam thinking de raspuns
    if "</think>" in response:
        think_part = response.split("</think>")[0].replace("<think>", "").strip()
        answer_part = response.split("</think>")[1].strip()
        print(f"🧠 Thinking: {think_part[:200]}...")
        print(f"💡 {answer_part[:300]}")
    else:
        print(f"💡 {response[:400]}")
    print("-" * 60)

## 9. Download adapterii

Descarcă folderul `smartbac-lora-final` din Output.
Conține:
- `adapter_model.safetensors` (~50MB) - greutățile LoRA
- `adapter_config.json` - configurația LoRA

Pune-le în proiect la `ai/finetune/adapters/`

In [ ]:
# Zip pentru download usor
import shutil
shutil.make_archive("/kaggle/working/smartbac-lora-final", "zip", FINAL_DIR)
print("\n📦 Archiva creata: /kaggle/working/smartbac-lora-final.zip")
print("Descarca din Output tab din dreapta ->")

## 10. (Optional) Push pe Hugging Face Hub

In [ ]:
# Decomentează dacă vrei să publici pe HF Hub
# from huggingface_hub import login
# login(token="hf_YOUR_TOKEN_HERE")
# model.push_to_hub("YOUR_USERNAME/smartbac-qwen-lora")
# tokenizer.push_to_hub("YOUR_USERNAME/smartbac-qwen-lora")
# print("Publicat pe Hugging Face Hub!")